In [1]:
requests = [
    {"id": "A", "prompt_tokens": 5, "decode_tokens": 6},
    {"id": "B", "prompt_tokens": 12, "decode_tokens": 4},
    {"id": "C", "prompt_tokens": 3, "decode_tokens": 3},
    {"id": "D", "prompt_tokens": 8, "decode_tokens": 8},
]

In [2]:
def static_batching(requests):
    time = 0

    max_prompt = max(r["prompt_tokens"] for r in requests)
    max_decode = max(r["decode_tokens"] for r in requests)

    print("Static batching")

    for t in range(max_prompt):
        active = [r["id"] for r in requests if t < r["prompt_tokens"]]
        print(f"time {time}: prefill active={active}")
        time += 1

    for t in range(max_decode):
        active = [r["id"] for r in requests if t < r["decode_tokens"]]
        print(f"time {time}: decode active={active}")
        time += 1

static_batching(requests)

Static batching
time 0: prefill active=['A', 'B', 'C', 'D']
time 1: prefill active=['A', 'B', 'C', 'D']
time 2: prefill active=['A', 'B', 'C', 'D']
time 3: prefill active=['A', 'B', 'D']
time 4: prefill active=['A', 'B', 'D']
time 5: prefill active=['B', 'D']
time 6: prefill active=['B', 'D']
time 7: prefill active=['B', 'D']
time 8: prefill active=['B']
time 9: prefill active=['B']
time 10: prefill active=['B']
time 11: prefill active=['B']
time 12: decode active=['A', 'B', 'C', 'D']
time 13: decode active=['A', 'B', 'C', 'D']
time 14: decode active=['A', 'B', 'C', 'D']
time 15: decode active=['A', 'B', 'D']
time 16: decode active=['A', 'D']
time 17: decode active=['A', 'D']
time 18: decode active=['D']
time 19: decode active=['D']


In [3]:
def continuous_batching(requests):
    active = []
    waiting = requests.copy()
    finished = []
    time = 0

    print("Continuous batching")

    while waiting or active:
        # Add one new request per step if available
        if waiting:
            r = waiting.pop(0)
            r = r.copy()
            r["phase"] = "prefill"
            active.append(r)

        print(f"\ntime {time}")

        for r in active:
            if r["phase"] == "prefill":
                r["prompt_tokens"] -= 1
                print(f"request {r['id']}: prefill")

                if r["prompt_tokens"] == 0:
                    r["phase"] = "decode"

            elif r["phase"] == "decode":
                r["decode_tokens"] -= 1
                print(f"request {r['id']}: decode")

        still_active = []
        for r in active:
            if r["decode_tokens"] == 0:
                print(f"request {r['id']}: finished")
                finished.append(r)
            else:
                still_active.append(r)

        active = still_active
        time += 1

continuous_batching(requests)

Continuous batching

time 0
request A: prefill

time 1
request A: prefill
request B: prefill

time 2
request A: prefill
request B: prefill
request C: prefill

time 3
request A: prefill
request B: prefill
request C: prefill
request D: prefill

time 4
request A: prefill
request B: prefill
request C: prefill
request D: prefill

time 5
request A: decode
request B: prefill
request C: decode
request D: prefill

time 6
request A: decode
request B: prefill
request C: decode
request D: prefill

time 7
request A: decode
request B: prefill
request C: decode
request D: prefill
request C: finished

time 8
request A: decode
request B: prefill
request D: prefill

time 9
request A: decode
request B: prefill
request D: prefill

time 10
request A: decode
request B: prefill
request D: prefill
request A: finished

time 11
request B: prefill
request D: decode

time 12
request B: prefill
request D: decode

time 13
request B: decode
request D: decode

time 14
request B: decode
request D: decode

time 15
requ

In [4]:
def chunked_prefill(prompt_tokens, chunk_size):
    remaining = prompt_tokens
    step = 0

    while remaining > 0:
        chunk = min(chunk_size, remaining)
        remaining -= chunk

        print(f"step {step}: process prefill chunk of {chunk} tokens")
        print(f"remaining prompt tokens: {remaining}")

        step += 1

    print("prefill done, decode can start")

chunked_prefill(prompt_tokens=20, chunk_size=6)

step 0: process prefill chunk of 6 tokens
remaining prompt tokens: 14
step 1: process prefill chunk of 6 tokens
remaining prompt tokens: 8
step 2: process prefill chunk of 6 tokens
remaining prompt tokens: 2
step 3: process prefill chunk of 2 tokens
remaining prompt tokens: 0
prefill done, decode can start


In [5]:
prompt1 = "You are a helpful AI inference tutor. Explain KV cache."
prompt2 = "You are a helpful AI inference tutor. Explain prefill."

prefix = "You are a helpful AI inference tutor."

print("prompt 1:", prompt1)
print("prompt 2:", prompt2)
print("shared prefix:", prefix)

print("\nWithout prefix cache:")
print("compute full prompt 1")
print("compute full prompt 2")

print("\nWith prefix cache:")
print("compute shared prefix once")
print("reuse prefix KV cache")
print("only compute remaining part of prompt 1 and prompt 2")

prompt 1: You are a helpful AI inference tutor. Explain KV cache.
prompt 2: You are a helpful AI inference tutor. Explain prefill.
shared prefix: You are a helpful AI inference tutor.

Without prefix cache:
compute full prompt 1
compute full prompt 2

With prefix cache:
compute shared prefix once
reuse prefix KV cache
only compute remaining part of prompt 1 and prompt 2
